# 09c - Does the patient's history BEFORE admission predict prolonged stays? (run_02)

Everything here ends before the admission time, so it is all allowed for an admission-time prediction:
- **Visit history**: number of earlier inpatient stays (all time and last 365 days), days since the last one,
  length of earlier stays (last, mean, max), earlier prolonged stays, earlier stays with the same admission reason,
  and outpatient, emergency and wellness visits in the last 365 days.
- **Procedure history**: procedures in the 365 days before admission (total, plus counts of the 40 most common).
- **Medication history**: medications started in the 365 days before admission (total, plus the 40 most common),
  and the number active at admission.

Same classifier and patient split as notebooks 08 and 09a; average precision (AP) on test with a paired patient
bootstrap against the tabular baseline (0.135).

Runtime: CPU, about 10-20 minutes (procedures and medications are read in chunks).

In [ ]:
# Cell 1 - load the stays and the notebook-01 split
import os, sys, json, time, numpy as np, pandas as pd, warnings
warnings.filterwarnings("ignore")
if "google.colab" in sys.modules:
    from google.colab import drive; drive.mount('/content/drive')
    BASE = "/content/drive/MyDrive/genai_synthea/run_02"
else:
    BASE = os.environ.get("RUN_BASE", "run_02")
RAW, OUT = f"{BASE}/raw/csv", f"{BASE}/features_history"
os.makedirs(OUT, exist_ok=True)
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score, roc_auc_score
meta = json.load(open(f"{BASE}/genai/meta.json")); THR = meta["plos_threshold"]
tab = pd.concat([pd.read_parquet(f"{BASE}/genai/{s}.parquet").assign(split=s) for s in ("train", "val", "test")], ignore_index=True)
tab["is_plos"] = (tab.target > THR).astype(int)
E = pd.read_csv(f"{RAW}/encounters.csv", usecols=["Id", "PATIENT", "START", "STOP", "ENCOUNTERCLASS", "REASONDESCRIPTION"])
for c in ("START", "STOP"): E[c] = pd.to_datetime(E[c], errors="coerce", utc=True).dt.tz_localize(None)
stays = tab[["hadm_id", "subject_id", "split", "target", "is_plos"]].merge(
    E[["Id", "START", "REASONDESCRIPTION"]].rename(columns={"Id": "hadm_id", "START": "admittime", "REASONDESCRIPTION": "reason"}), on="hadm_id")
subjects = set(stays.subject_id); CHUNK = 200_000; YEAR = pd.Timedelta(days=365)
print(f"Stays {len(stays):,} | patients {len(subjects):,} | PLOS {stays.is_plos.mean():.1%}")

In [ ]:
# Cell 2 - visit history (encounters that ENDED before this admission)
P = E[E.PATIENT.isin(subjects)].rename(columns={"PATIENT": "subject_id"})
P["los"] = (P.STOP - P.START).dt.total_seconds() / 86400
m = stays[["hadm_id", "subject_id", "admittime", "reason"]].merge(P, on="subject_id")
m = m[(m.STOP <= m.admittime) & (m.Id != m.hadm_id)]
inp = m[m.ENCOUNTERCLASS == "inpatient"].sort_values("START")
recent = m[m.START >= m.admittime - YEAR]
g = inp.groupby("hadm_id")
H = pd.DataFrame({
    "prior_inp_all": g.size(),
    "prior_inp_365": inp[inp.START >= inp.admittime - YEAR].groupby("hadm_id").size(),
    "days_since_inp": (g.admittime.first() - g.STOP.max()).dt.total_seconds() / 86400,
    "last_inp_los": g.los.last(), "mean_prior_los": g.los.mean(), "max_prior_los": g.los.max(),
    "prior_plos": inp[inp.los > THR].groupby("hadm_id").size(),
    "prior_same_reason": inp[inp.reason == inp.REASONDESCRIPTION].groupby("hadm_id").size(),
    "last_same_reason_los": inp[inp.reason == inp.REASONDESCRIPTION].groupby("hadm_id").los.last(),
})
for cls in ["ambulatory", "outpatient", "emergency", "wellness", "urgentcare"]:
    H[f"{cls}_365"] = recent[recent.ENCOUNTERCLASS == cls].groupby("hadm_id").size()
H = H.reindex(stays.hadm_id)
count_cols = [c for c in H if c not in ("days_since_inp", "last_inp_los", "mean_prior_los", "max_prior_los", "last_same_reason_los")]
H[count_cols] = H[count_cols].fillna(0)
H["days_since_inp"] = H["days_since_inp"].fillna(9999)          # no earlier stay
H = H.fillna(0); VISIT = list(H.columns)
print("Visit-history features:", VISIT)
print(H.join(stays.set_index("hadm_id").is_plos).groupby("is_plos")[["prior_inp_365", "last_inp_los", "prior_plos", "prior_same_reason"]].mean().round(2))

In [ ]:
# Cell 3 - procedure and medication history in the 365 days before admission (chunked)
key = stays[["hadm_id", "subject_id", "admittime"]]
train_ids = set(stays.loc[stays.split == "train", "hadm_id"])
def history(path, cols, code_col, active=False):
    if os.path.exists(f"{OUT}/{os.path.basename(path)}.pkl"): return pd.read_pickle(f"{OUT}/{os.path.basename(path)}.pkl")
    parts, t0 = [], time.time()
    for c in pd.read_csv(path, usecols=cols, chunksize=CHUNK):
        c = c[c.PATIENT.isin(subjects)].rename(columns={"PATIENT": "subject_id"})
        if c.empty: continue
        c["START"] = pd.to_datetime(c.START, errors="coerce", utc=True).dt.tz_localize(None)
        if active: c["STOP"] = pd.to_datetime(c.STOP, errors="coerce", utc=True).dt.tz_localize(None)
        x = c.merge(key, on="subject_id")
        before = x.START < x.admittime
        win = before & (x.START >= x.admittime - YEAR)
        agg = x[win].groupby(["hadm_id", code_col]).size().rename("n").reset_index()
        if active:
            act = x[before & (x.STOP.isna() | (x.STOP >= x.admittime))].groupby("hadm_id").size()
            agg = pd.concat([agg, act.rename("n").reset_index().assign(**{code_col: "__ACTIVE__"})])
        parts.append(agg)
    out = pd.concat(parts).groupby(["hadm_id", code_col]).n.sum().reset_index()
    out.to_pickle(f"{OUT}/{os.path.basename(path)}.pkl"); print(f"  {os.path.basename(path)}: {time.time()-t0:.0f}s")
    return out
def wide(long, code_col, prefix, k=40):
    top = (long[long.hadm_id.isin(train_ids) & (long[code_col] != "__ACTIVE__")]
           .groupby(code_col).n.sum().sort_values(ascending=False).head(k).index)       # chosen on TRAIN only
    W = long[long[code_col].isin(top)].pivot_table(index="hadm_id", columns=code_col, values="n", aggfunc="sum")
    W.columns = [f"{prefix}_{c}" for c in W.columns]
    W[f"{prefix}_total_365"] = long[long[code_col] != "__ACTIVE__"].groupby("hadm_id").n.sum()
    if (long[code_col] == "__ACTIVE__").any():
        W[f"{prefix}_active_at_admit"] = long[long[code_col] == "__ACTIVE__"].set_index("hadm_id").n
    return W.reindex(stays.hadm_id).fillna(0)
PR = wide(history(f"{RAW}/procedures.csv", ["START", "PATIENT", "CODE"], "CODE"), "CODE", "proc")
MD = wide(history(f"{RAW}/medications.csv", ["START", "STOP", "PATIENT", "CODE"], "CODE", active=True), "CODE", "med")
PROC_H, MED_H = list(PR.columns), list(MD.columns)
HIST = pd.concat([H, PR, MD], axis=1); HIST.index.name = "hadm_id"
HIST.reset_index().to_parquet(f"{OUT}/history_features.parquet", index=False)
print(f"Procedure-history features {len(PROC_H)} | medication-history features {len(MED_H)} | saved history_features.parquet")

In [ ]:
# Cell 4 - does history help? AP on test, paired patient bootstrap vs the tabular baseline
TAB_COLS = meta["num_cols"] + meta.get("cat_cols", []) + meta.get("bin_cols", [])
D = stays.merge(tab[["hadm_id"] + TAB_COLS], on="hadm_id").merge(HIST.reset_index(), on="hadm_id")
def X_of(d, extra): return pd.concat([pd.get_dummies(d[TAB_COLS], columns=meta.get("cat_cols", []), dtype=float).reset_index(drop=True),
                                      d[extra].astype(float).reset_index(drop=True)], axis=1)
SETS = {"A. tabular (baseline)": [], "A + visit history": VISIT, "A + procedure history": PROC_H,
        "A + medication history": MED_H, "A + all history": VISIT + PROC_H + MED_H}
tr, va, te = (D[D.split == s].reset_index(drop=True) for s in ("train", "val", "test"))
yte = te.is_plos.values; rng = np.random.default_rng(0); NB = int(os.environ.get("NBOOT", 1000))
pats = te.subject_id.values; up = np.unique(pats); pos = {p: np.where(pats == p)[0] for p in up}
boots = [np.concatenate([pos[p] for p in rng.choice(up, len(up))]) for _ in range(NB)]
preds, rows = {}, []
for name, extra in SETS.items():
    Xtr = X_of(tr, extra)
    m = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, random_state=0).fit(Xtr, tr.is_plos)
    pv = m.predict_proba(X_of(va, extra).reindex(columns=Xtr.columns, fill_value=0))[:, 1]
    pt = m.predict_proba(X_of(te, extra).reindex(columns=Xtr.columns, fill_value=0))[:, 1]
    preds[name] = pt
    rows.append({"features": name, "n_features": Xtr.shape[1], "AP_val": average_precision_score(va.is_plos, pv),
                 "AP_test": average_precision_score(yte, pt), "AUROC_test": roc_auc_score(yte, pt)})
base = preds["A. tabular (baseline)"]
for r in rows:
    gains = [average_precision_score(yte[i], preds[r["features"]][i]) - average_precision_score(yte[i], base[i])
             for i in boots if yte[i].sum() > 0]
    r["AP_gain"] = r["AP_test"] - average_precision_score(yte, base)
    r["ci_low"], r["ci_high"] = np.percentile(gains, [2.5, 97.5])
res = pd.DataFrame(rows).set_index("features"); res.to_csv(f"{OUT}/09c_history_utility.csv")
print(f"Test PLOS prevalence {yte.mean():.3f} (AP of a random model)")
res.round(4)

In [ ]:
# Cell 5 - which history features matter most (permutation importance on test, best set)
from sklearn.inspection import permutation_importance
extra = VISIT + PROC_H + MED_H
Xtr, Xte = X_of(tr, extra), X_of(te, extra).reindex(columns=X_of(tr, extra).columns, fill_value=0)
m = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, random_state=0).fit(Xtr, tr.is_plos)
pi = permutation_importance(m, Xte, yte, scoring="average_precision", n_repeats=5, random_state=0)
imp = pd.Series(pi.importances_mean, index=Xte.columns).sort_values(ascending=False)
print("Top 15 features by drop in test AP when shuffled:")
imp.head(15).round(4)